In [ ]:
# Step 1: Install required packages
!pip install ultralytics --upgrade -q
!pip install opencv-python-headless -q  # Ensure OpenCV is installed

# Step 2: Import required modules
import os
import zipfile
import shutil
from google.colab import drive
from ultralytics import YOLO

# Step 3: Mount Google Drive
drive.mount('/content/drive')

# Step 4: Define Paths in Google Drive
gdrive_base_path = "/content/drive/MyDrive/yolo_furniture_results"  # Google Drive storage for results
dataset_zip_path = "/content/drive/MyDrive/furnitures.zip"  # Path to dataset ZIP in Google Drive
dataset_extracted_path = "/content/furnitures"  # Temp extract location
yolo_dataset_path = f"{gdrive_base_path}/dataset"  # Final dataset path in Google Drive

# Step 5: Ensure Clean Dataset Extraction
if os.path.exists(dataset_extracted_path):
    shutil.rmtree(dataset_extracted_path)  # Remove old extracted data
os.makedirs(dataset_extracted_path, exist_ok=True)  # Create fresh folder

# Step 6: Extract Dataset from ZIP
print("📦 Extracting dataset...")
with zipfile.ZipFile(dataset_zip_path, 'r') as zip_ref:
    zip_ref.extractall(dataset_extracted_path)  # Extract ZIP to dataset path
print("✅ Dataset extracted successfully!")

# Step 7: Fix Nested Dataset Issue (If "furnitures/furnitures/" Exists)
if os.path.exists(f"{dataset_extracted_path}/furnitures"):
    nested_path = f"{dataset_extracted_path}/furnitures"
    for item in os.listdir(nested_path):
        shutil.move(os.path.join(nested_path, item), dataset_extracted_path)  # Move all files to correct location
    shutil.rmtree(nested_path)  # Remove extra folder

# Step 8: Verify Dataset Structure
expected_folders = ["train", "test"]
for category in ["chair", "sofa", "bed", "table", "tv"]:
    for folder in expected_folders:
        if not os.path.exists(f"{dataset_extracted_path}/{folder}/{category}/images") or \
           not os.path.exists(f"{dataset_extracted_path}/{folder}/{category}/labels"):
            print(f"❌ Error: Missing '{category}/images' or '{category}/labels' in '{folder}/'.")
            exit(1)

print("✅ Dataset folder structure is correct!")

# Step 9: Move Dataset to Google Drive for Training
if os.path.exists(yolo_dataset_path):
    shutil.rmtree(yolo_dataset_path)  # Remove old dataset in Google Drive
shutil.move(dataset_extracted_path, yolo_dataset_path)  # Move dataset to Google Drive

print("✅ Dataset moved to Google Drive at:", yolo_dataset_path)

# Step 10: Create YOLO Dataset Configuration File
yaml_path = f"{yolo_dataset_path}/data.yaml"

with open(yaml_path, 'w') as f:
    f.write(f"""
path: {yolo_dataset_path}
train: {yolo_dataset_path}/train
val: {yolo_dataset_path}/test
nc: 5
names: ['chair','table','bed','tv','sofa']
""")

print(f"✅ YOLO dataset configuration file created at: {yaml_path}")

# Step 11: Load YOLOv8 Model
model = YOLO("yolov8n.pt")  # Using YOLOv8 Nano model for faster training

# Step 12: Define Training Parameters
epochs = 20
batch_size = 16
image_size = 640
workers = 2

# Step 13: Train the Model and Save Everything in Google Drive
print("🚀 Training YOLOv8 on Furniture Dataset...")
model.train(
    data=yaml_path,         # Path to dataset configuration
    epochs=epochs,          # Number of epochs
    batch=batch_size,       # Batch size
    imgsz=image_size,       # Image size
    workers=workers,        # Number of workers
    project=gdrive_base_path,  # Save results in Google Drive
    name="furniture_training",  # Subfolder name
    pretrained=True,        # Use pretrained weights
    val=True,               # Enable validation
    device="cuda"           # Use GPU if available
)

# Step 14: Save Trained Model in Google Drive
trained_model_path = f"{gdrive_base_path}/furniture_training/weights/best.pt"

if os.path.exists(trained_model_path):
    print(f"✅ Model training completed. Model saved at: {trained_model_path}")
else:
    print("❌ Error: Trained model not found! Please check training logs.")

print("✅ All results and outputs are stored in Google Drive.")

